# Diabetic Retinopathy Stage Detection with Transfer Learning

**Module:** Computer Vision - BSc (Hons) Computer Science, NIBM / Coventry University
**Coursework 1 - Diabetic Retinopathy Stage Detection**

This notebook runs the complete pipeline on a free Google Colab GPU:

| Step | Rubric criterion | Section |
|---|---|---|
| Dataset download + exploration | 1. Problem understanding & dataset | 2 - 3 |
| Border crop, resize, CLAHE, Ben Graham, edge enhancement | 2. Preprocessing | 4 |
| On-the-fly augmentation + class weights | 3. Augmentation & balancing | 5 |
| EfficientNet / ResNet / MobileNet / DenseNet transfer learning | 4. CNN architecture | 6 |
| Two-phase training, early stopping, LR scheduling, checkpoints | 5. Training strategy | 6 |
| Accuracy/loss curves, precision, recall, F1, QWK, confusion matrix, ROC, Grad-CAM | 6. Evaluation | 7 |
| Backbone and preprocessing comparison | 5 + 9. Experimental design / innovation | 8 |
| Export results bundle for the report + Streamlit app | 7 - 8. Code & report | 9 |

> **Before you start:** `Runtime -> Change runtime type -> T4 GPU`.
> The whole notebook takes ~10 min for the main model and ~60 min if the full comparison grid is enabled.

All the logic lives in the `dr_detection` Python package of the repository (so the code is
modular, commented and re-usable by the command-line scripts and the Streamlit demo);
this notebook only *orchestrates* it and displays the evidence.

## 1. Environment setup

In [ ]:
# ============================================================================
# 1.1  Get the code base.  Set REPO_URL to your GitHub repository.
#      (Fallback: upload a zip of the repository if the repo is private.)
# ============================================================================
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/<your-username>/diabetic-retinopathy-stage-detection.git"   # <-- EDIT
REPO_DIR = "/content/diabetic-retinopathy-stage-detection"

if not pathlib.Path(REPO_DIR).exists():
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:\n", r.stderr)
        print("Upload the repository zip instead (File -> Upload), then unzip it to", REPO_DIR)
        from google.colab import files
        up = files.upload()
        zip_name = next(iter(up))
        subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "/content/_repo"], check=True)
        # the zip may contain one top-level folder
        inner = [p for p in pathlib.Path("/content/_repo").iterdir() if p.is_dir()]
        os.rename(str(inner[0]) if len(inner) == 1 else "/content/_repo", REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print("Repository:", REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout or "(no git history)")

In [ ]:
# ============================================================================
# 1.2  Install the few packages Colab does not ship with and check the GPU
# ============================================================================
!pip install -q kaggle "opencv-python-headless>=4.8" seaborn tqdm

import tensorflow as tf, keras, cv2, sklearn, numpy as np, pandas as pd
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| OpenCV", cv2.__version__, "| scikit-learn", sklearn.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - switch the runtime to a GPU!")
if gpus:
    print(tf.config.experimental.get_device_details(gpus[0]).get("device_name"))

## 2. Dataset: APTOS 2019 Blindness Detection (224 x 224 version)

**Source:** Kaggle - `sovitrath/diabetic-retinopathy-224x224-2019-data` (CC0 licence).
It is the training set of the *APTOS 2019 Blindness Detection* competition (Aravind Eye Hospital,
India), resized by the dataset author from ~3000 px to 224 x 224 px: **3,662 colour fundus photographs**
graded by clinicians on the International Clinical DR scale:

| Grade | Folder | Meaning |
|---|---|---|
| 0 | `No_DR` | no diabetic retinopathy |
| 1 | `Mild` | microaneurysms only |
| 2 | `Moderate` | more than microaneurysms, less than severe |
| 3 | `Severe` | > 20 haemorrhages per quadrant, venous beading, IRMA |
| 4 | `Proliferate_DR` | neovascularisation / vitreous haemorrhage |

Download requires a Kaggle API token (`kaggle.json` from *Kaggle -> Settings -> API*).
The token is stored only in this Colab session and is never printed.

In [ ]:
# ============================================================================
# 2.1  Kaggle credentials -> download -> unzip   (skipped if already present)
# ============================================================================
DATA_DIR = "/content/data/diabetic_retinopathy_data"

if not pathlib.Path(DATA_DIR, "colored_images").exists():
    if not pathlib.Path("/root/.kaggle/kaggle.json").exists():
        from google.colab import files
        print("Upload your kaggle.json:")
        files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !python scripts/download_data.py --out "$DATA_DIR"
else:
    print("dataset already present")
!du -sh "$DATA_DIR" && ls "$DATA_DIR" && ls "$DATA_DIR/colored_images"

In [ ]:
# ============================================================================
# 2.2  Configuration of the MAIN experiment (all hyper-parameters in one place)
# ============================================================================
from dr_detection import config as C
from dr_detection.utils import set_seed

cfg = C.Config(
    data_dir=DATA_DIR,
    results_dir=f"{REPO_DIR}/results",
    models_dir=f"{REPO_DIR}/models",
    cache_dir="/content/data/cache",
    backbone="EfficientNetB0",      # main model (see section 8 for alternatives)
    preprocess="ben_graham",        # main preprocessing (see section 8 for alternatives)
    img_size=224, batch_size=32,
    epochs_phase1=10, epochs_phase2=20,
    lr_phase1=1e-4, lr_phase2=1e-5, unfreeze_layers=30,
    dropout=0.3, use_class_weights=True, use_augmentation=True,
    early_stopping_patience=6, reduce_lr_patience=3, seed=42,
)
set_seed(cfg.seed)
FIG_DIR = pathlib.Path(cfg.results_dir) / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(cfg.to_dict().items(), columns=["parameter", "value"])

## 3. Exploratory data analysis

In [ ]:
# ============================================================================
# 3.1  Build the DataFrame (robust label mapping: NO images are dropped)
# ============================================================================
from dr_detection.data import build_dataframe, cross_check_with_csv, class_distribution, split_dataframe

df = build_dataframe(cfg.images_dir)
df = cross_check_with_csv(df, cfg.data_dir)     # verifies folder labels against train.csv
print("Total images:", len(df))
class_distribution(df)

In [ ]:
# ============================================================================
# 3.2  Stratified 70 / 10 / 20 split (same seed for every experiment)
# ============================================================================
df_train, df_val, df_test = split_dataframe(df, cfg.val_frac, cfg.test_frac, cfg.seed)
pd.DataFrame({
    "train": df_train["label"].value_counts().sort_index(),
    "validation": df_val["label"].value_counts().sort_index(),
    "test": df_test["label"].value_counts().sort_index(),
}).rename(index=lambda i: f"{i} - {C.CLASS_LABELS[i]}").assign(total=lambda t: t.sum(axis=1))

In [ ]:
# ============================================================================
# 3.3  Figures: class distribution, split, samples, illumination variability
# ============================================================================
from dr_detection import visualize as V
from IPython.display import Image as IImage, display

V.plot_class_distribution(df, FIG_DIR / "class_distribution.png")
V.plot_split_distribution(df_train, df_val, df_test, FIG_DIR / "split_distribution.png")
V.plot_sample_grid(df, FIG_DIR / "sample_grid_raw.png", method="raw", seed=cfg.seed)
stats = V.plot_intensity_stats(df, FIG_DIR / "intensity_stats.png", seed=cfg.seed)
for f in ["class_distribution.png", "split_distribution.png", "sample_grid_raw.png", "intensity_stats.png"]:
    display(IImage(filename=str(FIG_DIR / f), width=800))

## 4. Preprocessing

Every image goes through: **black-border crop -> resize 224 px -> contrast / edge enhancement**.

* **Ben Graham** (`4*img - 4*GaussianBlur(img, sigma=10) + 128`): removes illumination gradients and
  colour casts between cameras and amplifies local contrast, so micro-aneurysms, haemorrhages and
  exudates stand out. A circular mask removes the artificial grey frame it creates.
* **CLAHE** on the L channel of LAB: adaptive local contrast without colour shifts.
* **Unsharp mask / Canny edge map**: shown to illustrate what *edge enhancement* emphasises (vessels).

Section 8 measures which method actually helps the classifier.

In [ ]:
# ============================================================================
# 4.1  One image through every stage, and every method per grade
# ============================================================================
example = df[df["label"] == 2].sample(1, random_state=cfg.seed)["image_path"].iloc[0]
V.plot_preprocessing_stages(example, FIG_DIR / "preprocessing_stages.png")
V.plot_preprocessing_comparison(df, FIG_DIR / "preprocessing_comparison.png", seed=cfg.seed)
V.plot_sample_grid(df, FIG_DIR / "sample_grid_ben_graham.png", method="ben_graham", seed=cfg.seed)
display(IImage(filename=str(FIG_DIR / "preprocessing_stages.png"), width=1000))
display(IImage(filename=str(FIG_DIR / "preprocessing_comparison.png"), width=700))

In [ ]:
# ============================================================================
# 4.2  Preprocess the whole dataset once into RAM (cached as .npz)
# ============================================================================
from dr_detection.data import prepare_data
data = prepare_data(cfg)
{k: (v.shape if hasattr(v, "shape") else len(v)) for k, v in data.items() if k.startswith(("X_", "y_"))}

## 5. Data augmentation and class balancing

Augmentation runs **on the GPU, on the fly, on the training split only**: random flips, rotation
(+/-36 deg), zoom (10 %), translation (5 %), brightness and contrast (10 %). Every epoch therefore
sees a different variant of each image, which fights over-fitting far better than a fixed set of copies.

The dataset is imbalanced (49 % No DR vs 5 % Severe). We use **inverse-frequency class weights**
in the loss (`weight_c = N / (5 * n_c)`), so a mistake on a rare severe case costs ~4x more than on a
healthy one. Random over-sampling is also implemented (`Config(oversample_minority=True)`).

In [ ]:
from dr_detection.augmentation import compute_class_weights
weights = compute_class_weights(data["y_train"])
V.plot_augmentation_grid(example, FIG_DIR / "augmentation_grid.png", method=cfg.preprocess)
V.plot_class_balance(data["y_train"], weights, FIG_DIR / "class_balance.png")
display(IImage(filename=str(FIG_DIR / "augmentation_grid.png"), width=1000))
display(IImage(filename=str(FIG_DIR / "class_balance.png"), width=700))
pd.DataFrame({"class": C.CLASS_LABELS, "train images": [int((data["y_train"]==c).sum()) for c in range(5)],
              "class weight": [round(weights[c], 3) for c in range(5)]})

## 6. CNN architecture, transfer learning and two-phase training

```
Input 224x224x3 (0-255) -> backbone normalisation -> EfficientNetB0 (ImageNet, include_top=False)
    -> GlobalAveragePooling -> Dropout 0.3 -> Dense 512 ReLU -> Dropout 0.3 -> Dense 256 ReLU -> Dense 5 softmax
```
* **Phase 1** - backbone frozen, head trained with Adam 1e-4 (up to 10 epochs)
* **Phase 2** - top 30 layers un-frozen (BatchNorm kept frozen), Adam 1e-5 (up to 20 epochs)
* Callbacks: `EarlyStopping(val_loss, patience 6, restore best)`, `ReduceLROnPlateau(x0.5, patience 3)`,
  `ModelCheckpoint`, `CSVLogger`
* Loss: sparse categorical cross-entropy with class weights

In [ ]:
# ============================================================================
# 6.1  Inspect the architecture before training
# ============================================================================
from dr_detection.model import build_model, count_params
_model, _base = build_model(cfg)
print("backbone layers:", len(_base.layers), "| params:", count_params(_model))
_model.summary(show_trainable=True)
del _model, _base

In [ ]:
# ============================================================================
# 6.2  Train + evaluate the MAIN model  (~5-10 min on a T4)
#      Writes results/<run_name>/ and models/best_model.keras for the app
# ============================================================================
from dr_detection.train import run_experiment
metrics = run_experiment(cfg, data=data, save_as_best=True)

## 7. Evaluation of the main model

In [ ]:
RUN = cfg.run_dir
hist = pd.read_csv(RUN / "history.csv")
print(f"Epochs trained: {len(hist)}  (phase 1: {(hist.phase==1).sum()}, phase 2: {(hist.phase==2).sum()})")
display(IImage(filename=str(RUN / "training_curves.png"), width=1000))
hist.round(4).tail(8)

In [ ]:
print(metrics["classification_report"])
summary = {k: metrics[k] for k in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
                                   "quadratic_weighted_kappa", "macro_auc_ovr", "within_one_grade_accuracy",
                                   "referable_sensitivity", "referable_specificity"] if k in metrics}
pd.DataFrame(summary.items(), columns=["metric", "test value"]).round(4)

In [ ]:
from IPython.display import HTML
for f in ["confusion_matrix.png", "confusion_matrix_normalised.png", "per_class_metrics.png", "roc_curves.png",
          "prediction_distribution.png", "misclassified.png", "gradcam_test_samples.png"]:
    display(HTML(f"<h4>{f}</h4>")); display(IImage(filename=str(RUN / f), width=800))

In [ ]:
# Referable-DR (grade >= 2) and any-DR (grade >= 1) screening performance
pd.DataFrame({"referable DR (>=2)": metrics["referable_dr"], "any DR (>=1)": metrics["any_dr"]}).round(4)

## 8. Experiments: preprocessing methods and CNN backbones

Same split, same seed, same training recipe - only one factor changes per run.
Set `RUN_COMPARISON = True` to execute (~45-60 min on a T4); results are collected into
`results/experiments_comparison.csv` / `.png`. Runs that already exist are skipped, so the cell can be re-run safely.

In [ ]:
RUN_COMPARISON = True          # <-- set False to skip the grid
PREPROCESS_GRID = ["raw", "clahe", "ben_graham", "clahe_ben_graham"]
BACKBONE_GRID   = ["EfficientNetB0", "EfficientNetB3", "ResNet50V2", "MobileNetV2", "DenseNet121"]

if RUN_COMPARISON:
    import dataclasses
    grid = [("EfficientNetB0", p) for p in PREPROCESS_GRID] + [(b, "ben_graham") for b in BACKBONE_GRID]
    grid = list(dict.fromkeys(grid))
    data_cache = {cfg.preprocess: data}
    for backbone, prep in grid:
        run_cfg = dataclasses.replace(cfg, backbone=backbone, preprocess=prep, run_name="")
        if (run_cfg.run_dir / "metrics.json").exists():
            print("skip (done):", run_cfg.run_name); continue
        if prep not in data_cache:
            data_cache[prep] = prepare_data(run_cfg)
        run_experiment(run_cfg, data=data_cache[prep], make_figures=True)
        tf.keras.backend.clear_session()

In [ ]:
import json
from dr_detection.visualize import plot_experiment_comparison
rows = []
for mfile in sorted(pathlib.Path(cfg.results_dir).glob("*/metrics.json")):
    m = json.load(open(mfile))
    rows.append({k: m.get(k) for k in ["run_name", "backbone", "preprocess", "accuracy", "macro_f1", "weighted_f1",
                                       "quadratic_weighted_kappa", "referable_sensitivity", "referable_specificity",
                                       "within_one_grade_accuracy", "epochs_trained", "train_minutes"]})
table = pd.DataFrame(rows).sort_values("quadratic_weighted_kappa", ascending=False)
table.to_csv(pathlib.Path(cfg.results_dir) / "experiments_comparison.csv", index=False)
plot_experiment_comparison(table, pathlib.Path(cfg.results_dir) / "experiments_comparison.png")
display(IImage(filename=str(pathlib.Path(cfg.results_dir) / "experiments_comparison.png"), width=1000))
table.round(4)

## 9. Export everything for the report and the Streamlit demo

In [ ]:
# ============================================================================
# 9.1  Copy 2 real test images per grade into app/assets/samples (for the demo video)
# ============================================================================
import shutil
samples_dir = pathlib.Path(REPO_DIR) / "app" / "assets" / "samples"; samples_dir.mkdir(parents=True, exist_ok=True)
for f in samples_dir.glob("*TEMP*"): f.unlink()
for c in range(5):
    for _, row in data["df_test"][data["df_test"]["label"] == c].sample(2, random_state=1).iterrows():
        shutil.copy(row["image_path"], samples_dir / f"grade{c}_{C.CLASS_NAMES[c]}_{row['id_code']}.png")
sorted(p.name for p in samples_dir.iterdir())

In [ ]:
# ============================================================================
# 9.2  Zip results/ + models/ + samples and download it  (also copied to Drive if mounted)
# ============================================================================
!python scripts/export_results.py --out /content/dr_results_bundle.zip
import zipfile
with zipfile.ZipFile("/content/dr_results_bundle.zip", "a") as zf:
    for p in samples_dir.iterdir():
        zf.write(p, p.relative_to(REPO_DIR))
print("bundle size: %.1f MB" % (os.path.getsize("/content/dr_results_bundle.zip") / 1e6))

try:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy("/content/dr_results_bundle.zip", "/content/drive/MyDrive/dr_results_bundle.zip")
    print("copied to Google Drive: MyDrive/dr_results_bundle.zip")
except Exception as e:
    print("Drive not mounted:", e)

from google.colab import files
files.download("/content/dr_results_bundle.zip")

---
**Next steps outside Colab**

1. Unzip `dr_results_bundle.zip` into the repository root (it recreates `results/`, `models/` and `app/assets/samples/`).
2. `pip install -r requirements.txt` and `streamlit run app/streamlit_app.py` to launch the demo used in the video.
3. All numbers and figures in the report come from `results/<run_name>/metrics.json` and the PNG files.